# Neuron-type composition by animal and headstage
Count the recorded units in the **same saved sessions used by the PCA and UMAP analyses**, grouped by animal and headstage. Inspect pooled counts, proportions and session-level composition to help contextualize differences in performance.

Match recorded neuron IDs **exactly** to `df_clus_encoding_phd.pkl`: Motor=0, Sensory=1, Sensorimotor=2, None=3. **Unclassified** means no matching classification; it is never included in None. Counts describe session-specific recorded units, not identified neurons tracked across days. Tracking/playback and raw/baseline-subtracted analyses use the same neuron set within a session, so units are counted once per session, not once per condition or activity mode.

This notebook is standalone: no PCA/UMAP fits or spike-count loading are needed. Run top to bottom with the shared `.venv` kernel. The saved session table determines the scope (currently AC recordings); absent animal/headstage combinations are not treated as zero recordings. Composition alone does not establish the cause of performance differences.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
repo_dir = next((candidate for parent in (cwd, *cwd.parents)
                 for candidate in (parent, parent / "etheremin_project")
                 if (candidate / "movement_bouts" / "evaluation.py").is_file()), None)
if repo_dir is None:
    raise FileNotFoundError("Run from the eTheremin workspace or inside the repository.")
sys.path.insert(0, str(repo_dir / "movement_bouts"))
from importlib import reload
import evaluation
reload(evaluation)

DATA_DIR = repo_dir.parent / "data"
SESSION_INDICES = None  # None = all saved sessions; or a list of original idx values.
SAVE_RESULTS = True
EXTRACTION_SUBDIR = "extracted_-1.2_0"
TYPE_COLUMNS = ["Motor", "Sensory", "Sensorimotor", "None", "Unclassified"]
TYPE_COLORS = {**evaluation.NEURON_TYPE_COLORS, "Unclassified": "white"}
CATEGORY_TO_CLUSTER = {"Motor": 0, "Sensory": 1, "Sensorimotor": 2, "None": 3}


## Load and check identity
Use the saved neuron-axis ID table, preserving session indices and original neuron order. Verify that the session metadata matches the extraction. No neuron-count cutoff is applied, and unclassified units remain in every denominator.


In [ ]:
session_path = DATA_DIR / "derived" / "movement_bouts" / "session_info.pkl"
extraction_dir = DATA_DIR / "derived" / "premotor_activity" / EXTRACTION_SUBDIR
ids_path = extraction_dir / "neuron_ids_by_session.pkl"
classification_path = DATA_DIR / "df_clus_encoding_phd.pkl"
extraction_parameters = json.loads((extraction_dir / "parameters.json").read_text(encoding="utf-8"))
if hashlib.sha256(session_path.read_bytes()).hexdigest() != extraction_parameters["source_sha256"]["session_info.pkl"]:
    raise ValueError("Session metadata changed since extraction; regenerate aligned neuron metadata first.")
session_info = pd.read_pickle(session_path)
neuron_ids_by_session = pd.read_pickle(ids_path)
classification_table = pd.read_pickle(classification_path)
if not session_info.index.is_unique or not neuron_ids_by_session.index.is_unique:
    raise ValueError("Session tables need unique original idx values.")
if SESSION_INDICES is not None:
    selected = pd.Index(SESSION_INDICES)
    if selected.empty or not selected.is_unique or not selected.isin(session_info.index).all():
        raise ValueError("Select nonempty, unique, existing session indices.")
    session_info = session_info.loc[session_info.index.isin(selected)].copy()
session_info = session_info.sort_index()
if session_info.empty or not session_info.index.isin(neuron_ids_by_session.index).all():
    raise ValueError("Every selected session must have saved neuron IDs.")
if classification_table["neuron_idx"].isna().any() or classification_table["neuron_idx"].duplicated().any():
    raise ValueError("Classification IDs must be unique and nonmissing.")
if not classification_table["cluster_encoding"].isin(CATEGORY_TO_CLUSTER.values()).all():
    raise ValueError("Unknown or missing classification code; inspect the classification source.")
cluster_by_id = classification_table.set_index("neuron_idx")["cluster_encoding"]
category_by_cluster = {value: key for key, value in CATEGORY_TO_CLUSTER.items()}
print(f"{len(session_info)} selected session entries; regions: {session_info['region'].unique().tolist()}")


## Count every recorded unit once per session
`neuron_inventory` retains each unit's original recording-axis position, ID and exact classification match. `session_counts` keeps the original session index and all session metadata for joining to saved performance summaries.


In [ ]:
inventory_rows = []
count_rows = []
for idx, info in session_info.iterrows():
    neuron_ids = pd.Index(neuron_ids_by_session.loc[idx, "neuron_ids"])
    if neuron_ids.hasnans or not neuron_ids.is_unique:
        raise ValueError(f"Missing or duplicated recorded neuron ID in session {idx}.")
    clusters = pd.Series(neuron_ids.map(cluster_by_id), index=neuron_ids)
    categories = clusters.map(category_by_cluster).fillna("Unclassified")
    counts = categories.value_counts().reindex(TYPE_COLUMNS, fill_value=0)
    count_rows.append({"idx": idx, **counts.to_dict(), "n_total": len(neuron_ids)})
    for axis_position, (neuron_id, category) in enumerate(categories.items()):
        inventory_rows.append({"idx": idx, "neuron_axis": axis_position, "neuron_id": neuron_id,
                               "neuron_type": category, "classified": category != "Unclassified"})
neuron_inventory = pd.DataFrame(inventory_rows, columns=["idx", "neuron_axis", "neuron_id", "neuron_type", "classified"])
session_counts = session_info.join(pd.DataFrame(count_rows).set_index("idx"), validate="one_to_one")
session_counts["n_classified"] = session_counts["n_total"] - session_counts["Unclassified"]
session_counts["classification_coverage_pct"] = 100 * session_counts["n_classified"] / session_counts["n_total"].replace(0, np.nan)
session_percentages = session_counts[TYPE_COLUMNS].div(session_counts["n_total"].replace(0, np.nan), axis=0) * 100
assert session_counts[TYPE_COLUMNS].sum(axis=1).eq(session_counts["n_total"]).all()
assert len(neuron_inventory) == session_counts["n_total"].sum()
assert not neuron_inventory.duplicated(["idx", "neuron_id"]).any()
repeated_ids = int(neuron_inventory["neuron_id"].duplicated().sum())
print(f"{len(neuron_inventory):,} recorded session-unit entries; {session_counts['n_classified'].sum():,} classified; {session_counts['Unclassified'].sum():,} unclassified.")
print(f"Repeated exact IDs across sessions: {repeated_ids}. Counts always use the (idx, neuron_id) identity.")
display(session_counts[["animal", "headstage", "session", *TYPE_COLUMNS, "n_total", "classification_coverage_pct"]])


## Check classification coverage by session
An entirely unclassified session contributes to all-neuron PCA/UMAP fits, but none of its units can enter a neuron-type-specific fit. This is missing classification coverage, not evidence that these units are in the None class. Check these rows before interpreting changes in type composition across recordings.

In [ ]:
entirely_unclassified_sessions = session_counts.loc[
    session_counts["n_total"].gt(0) & session_counts["n_classified"].eq(0),
    ["animal", "headstage", "session", "n_total"],
].copy()
print(f"{len(entirely_unclassified_sessions)} sessions have no matched classifications.")
display(entirely_unclassified_sessions)


## Animal/headstage totals and proportions
Totals sum **session-unit entries**. Pooled percentages weight each recorded unit equally; they can differ from the mean of per-session percentages when neuron counts vary. The block table also records the actual session indices instead of assuming fixed ranges.


In [ ]:
grouped = session_counts.groupby(["animal", "headstage"], sort=False, observed=True)
split_counts = grouped[TYPE_COLUMNS + ["n_total", "n_classified"]].sum()
split_counts.insert(0, "n_sessions", grouped.size())
split_counts.insert(1, "idx_first", grouped.apply(lambda rows: rows.index.min(), include_groups=False))
split_counts.insert(2, "idx_last", grouped.apply(lambda rows: rows.index.max(), include_groups=False))
split_counts["classification_coverage_pct"] = 100 * split_counts["n_classified"] / split_counts["n_total"].replace(0, np.nan)
split_session_indices = {key: rows.index.tolist() for key, rows in grouped}
split_percentages = split_counts[TYPE_COLUMNS].div(split_counts["n_total"].replace(0, np.nan), axis=0) * 100
mean_session_percentages = session_counts[["animal", "headstage"]].join(session_percentages).groupby(["animal", "headstage"], sort=False, observed=True)[TYPE_COLUMNS].mean()
animal_counts = session_counts.groupby("animal", sort=False, observed=True)[TYPE_COLUMNS + ["n_total"]].sum()
animal_counts.insert(0, "n_sessions", session_counts.groupby("animal", sort=False, observed=True).size())
assert split_counts[TYPE_COLUMNS].sum().equals(session_counts[TYPE_COLUMNS].sum())
assert np.allclose(split_percentages.sum(axis=1), 100)
print("Counts per animal/headstage:")
display(split_counts)
print("Pooled percentages (including unclassified units in denominator):")
display(split_percentages.round(1))
print("Mean per-session percentages (sessions weighted equally):")
display(mean_session_percentages.round(1))
print("Counts per animal, pooling its saved headstages:")
display(animal_counts)


## Compare recording blocks
White hatched bars are unclassified units; gray is the assigned None class. Numbers inside count bars are unit counts. Labels also report session counts, because pooled totals increase with both units per session and number of sessions.


In [ ]:
block_labels = [f"{animal} / hs{headstage}  (n={int(split_counts.loc[(animal, headstage), 'n_sessions'])} sessions)"
                for animal, headstage in split_counts.index]
fig_blocks, axes_blocks = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True, layout="constrained")
for ax, table, xlabel in zip(axes_blocks, (split_counts[TYPE_COLUMNS], split_percentages),
                            ("Recorded session-unit entries", "Percent of all recorded units")):
    left = np.zeros(len(table))
    for category in TYPE_COLUMNS:
        values = table[category].to_numpy(dtype=float)
        bars = ax.barh(np.arange(len(table)), values, left=left, color=TYPE_COLORS[category],
                       edgecolor=".3", linewidth=.5, hatch="///" if category == "Unclassified" else None,
                       label=category)
        if ax is axes_blocks[0]:
            ax.bar_label(bars, labels=[str(int(v)) if v else "" for v in values], label_type="center", fontsize=8)
        left += values
    ax.set(yticks=np.arange(len(table)), yticklabels=block_labels, xlabel=xlabel)
axes_blocks[0].invert_yaxis()
axes_blocks[0].set_title("Total neuron-type counts")
axes_blocks[1].set(title="Pooled composition", xlim=(0, 100))
axes_blocks[1].legend(title="Neuron type", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig_blocks.suptitle("Neuron-type composition by animal and headstage")
plt.show()


## Follow the session order used in the performance plots
The top panel shows total neuron count and type counts per session; the bottom shows fractions. Animal names and headstages use the same boundaries as the PCA/UMAP score plots. These counts can be joined to a saved score summary on `idx`; a large block total alone should not be interpreted as more neurons in each fitted model.


In [ ]:
fig_sessions, axes_sessions = plt.subplots(2, 1, figsize=(16, 9), sharex=True, layout="constrained")
for ax, table, ylabel in zip(axes_sessions, (session_counts[TYPE_COLUMNS], session_percentages),
                             ("Recorded units per session", "Percent of recorded units")):
    bottom = np.zeros(len(table))
    for category in TYPE_COLUMNS:
        values = table[category].fillna(0).to_numpy(dtype=float)
        ax.bar(table.index, values, bottom=bottom, width=.85, color=TYPE_COLORS[category],
               edgecolor=".3", linewidth=.25, hatch="///" if category == "Unclassified" else None,
               label=category)
        bottom += values
    ax.set(xlabel="Session entry (idx)", ylabel=ylabel)
    # Reuse the exact animal/headstage annotation used on score summaries.
    evaluation._annotate_session_animals(ax, session_counts.rename_axis("idx"))
axes_sessions[0].set_title("Neuron counts by session", pad=46)
axes_sessions[1].set_title("Composition by session", pad=46)
axes_sessions[0].set_ylim(0, max(1, session_counts["n_total"].max()) * 1.08)
axes_sessions[1].set_ylim(0, 100)
axes_sessions[0].legend(ncol=5, loc="upper center", bbox_to_anchor=(.5, -.13), frameon=False)
plt.show()


## Save the audit tables and figures
Pickle tables retain original session IDs for later performance comparisons. Source fingerprints and settings record exactly which classifications and recorded neuron IDs were counted. No PCA, UMAP or regression is fitted or changed.


In [ ]:
composition_settings = {
    "session_indices": [int(idx) for idx in session_info.index],
    "category_to_cluster": CATEGORY_TO_CLUSTER,
    "unclassified_policy": "No exact classification-ID match; separate from None",
    "counting_unit": "session-specific recorded neuron (idx, neuron_id)",
    "percentage_denominator": "all recorded units including unclassified",
    "source_sha256": {str(path): hashlib.sha256(path.read_bytes()).hexdigest()
                      for path in (session_path, ids_path, classification_path)},
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
composition_bundle = {
    "settings": composition_settings, "neuron_inventory": neuron_inventory,
    "session_counts": session_counts, "session_percentages": session_percentages,
    "split_counts": split_counts, "split_percentages": split_percentages,
    "mean_session_percentages": mean_session_percentages,
    "animal_counts": animal_counts, "entirely_unclassified_sessions": entirely_unclassified_sessions, "split_session_indices": split_session_indices,
}
if SAVE_RESULTS:
    effective_settings = {k: v for k, v in composition_settings.items() if k != "created_at_utc"}
    digest = hashlib.sha256(json.dumps(effective_settings, sort_keys=True).encode()).hexdigest()[:12]
    output_dir = DATA_DIR / "derived" / "neuron_type_composition" / digest
    output_dir.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(composition_bundle, output_dir / "composition.pkl")
    for name in ("session_counts", "session_percentages", "split_counts", "split_percentages", "mean_session_percentages", "animal_counts", "neuron_inventory", "entirely_unclassified_sessions"):
        composition_bundle[name].to_pickle(output_dir / f"{name}.pkl")
    (output_dir / "parameters.json").write_text(json.dumps(composition_settings, indent=2), encoding="utf-8")
    fig_blocks.savefig(output_dir / "animal_headstage_composition.png", dpi=180, bbox_inches="tight")
    fig_sessions.savefig(output_dir / "session_composition.png", dpi=180, bbox_inches="tight")
    print(f"Saved tables and figures to {output_dir}")
